# tinydistill quickstart

Turn an LLM classification prompt into a small, fast classifier, in a few minutes.

[tinydistill](https://github.com/Andreas-Hum/tinydistill) has an instruction-tuned LLM (the *teacher*) score
every label for every training text, trains a small encoder (the *student*) to imitate those soft labels, and
exports it to ONNX for fast CPU inference. No gold labels are needed.

This notebook distils the sentiment of financial news tweets (bearish, bullish, neutral) from
`Qwen/Qwen2.5-1.5B-Instruct` into `all-MiniLM-L6-v2` on 2,000 tweets. **Use a GPU runtime** (Runtime > Change
runtime type > T4 GPU); on a CPU the teacher is much slower.

In [ ]:
!pip install -q "tinydistill[hf]"

## 1. Describe the task

A task is a short YAML file: the prompt, with `{text}` where the input goes and `{labels}` for the list of
options, and the labels (each with the words the teacher should answer with). This one comes from the
repository's examples.

In [ ]:
!curl -sLO https://raw.githubusercontent.com/Andreas-Hum/tinydistill/main/examples/financial-tweets/task.yaml
print(open("task.yaml").read())

## 2. Label, train, export and evaluate

`tinydistill run` does every step: the teacher scores each label for each training text (one forward pass per
text, no text generation), the student trains on those soft labels, the student is exported to ONNX (and
int8), and both are evaluated on the held-out test split, which here has gold labels.

In [ ]:
!tinydistill run task.yaml \
  --data hf:zeroshot/twitter-financial-news-sentiment:train --limit 2000 \
  --test hf:zeroshot/twitter-financial-news-sentiment:validation --test-limit 500 \
  --teacher Qwen/Qwen2.5-1.5B-Instruct --int8 --out run

## 3. Results

Accuracy of the teacher and the student on the test texts, their agreement, latency and size. The student is
many times faster and smaller; how close it gets to the teacher depends mostly on the teacher.

In [ ]:
from IPython.display import Markdown

Markdown(open("run/results.md").read())

## 4. Use the student

The exported model needs only `onnxruntime` and a tokenizer, no PyTorch.

In [ ]:
from tinydistill import Predictor

clf = Predictor("run/onnx")  # uses the int8 model
texts = [
    "Shares jump 12% after the company beats earnings expectations",
    "Regulators open an investigation into the bank's lending practices",
    "The board will meet on Thursday to discuss the annual report",
]
for text, result in zip(texts, clf.describe(texts), strict=True):
    print(f"{result['label']:>8}  {result['confidence']:.2f}  {text}")

## 5. Where next

- **A stronger teacher** helps most: try `--teacher Qwen/Qwen2.5-7B-Instruct --load-4bit` (needs the `4bit`
  extra: `pip install "tinydistill[hf,4bit]"`).
- **Your own task:** `tinydistill init your_data.csv --out task.yaml` writes a starter task file from the
  labels in your data.
- **See what goes wrong:** `tinydistill analyse run` writes `run/analysis.html` with the weakest labels and the
  most common confusions.
- **Deploy:** `tinydistill serve run/onnx` (HTTP), `tinydistill web run/onnx --out site` (in the browser), or
  `tinydistill push run/onnx --repo you/name` (Hugging Face Hub, then `Predictor("hf:you/name")`).
- The [README](https://github.com/Andreas-Hum/tinydistill) has the full documentation and benchmarks.